In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import os
import seaborn as sns

In [2]:
os.getcwd()

'/Users/trixieadlaon/Projects/RGGI_VA/scripts'

## Data Load 

In [3]:
data_folder = '../data'
combined_df = []

for file in os.listdir(data_folder): 
    if file.endswith('.csv') and not file.startswith('final'): 
        full_path = os.path.join(data_folder, file) 
        try:
            df = pd.read_csv(full_path, encoding='utf-8', low_memory=False, skiprows=1) 
            
        except UnicodeDecodeError:
            df = pd.read_csv(full_path, encoding='latin-1', low_memory=False, skiprows=1) 
        
        combined_df.append(df)
            
master_df = pd.concat(combined_df, ignore_index=True)
master_df

,SEQPLT20,YEAR,PSTATABB,PNAME,ORISPL,OPRNAME,OPRCODE,UTLSRVNM,UTLSRVID,SECTOR,...,BIOCO2E,CHPCO2E,PLGENATO,PLGENACO,PLTOPR,PLCOPR,SEQPLT22,CAMDFLAG,SEQPLT,Unnamed: 141
0,1.0,2020,AK,7-Mile Ridge Wind Project,60814,Alaska Power and Telephone Co,219,Alaska Power and Telephone Co,219,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,2020,AK,Agrium Kenai Nitrogen Operations,54452,Homer Electric Assn Inc,19558,Agrium US Inc,179,Industrial CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,2020,AK,Alakanuk,57053,"Alaska Village Elec Coop, Inc",221,"Alaska Village Elec Coop, Inc",221,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,2020,AK,Allison Creek Hydro,58982,"Copper Valley Elec Assn, Inc",4329,"Copper Valley Elec Assn, Inc",4329,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,2020,AK,Ambler,60243,"Alaska Village Elec Coop, Inc",221,"Alaska Village Elec Coop, Inc",221,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71470,NaN,2021,WY,TransAlta Wyoming Wind,56093,PacifiCorp,14354,NextEra Energy Operating Services LLC,58528,IPP Non-CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11389.0,NaN
71471,NaN,2021,WY,Wygen I,55479,Black Hills Power Inc,19545,"Black Hills Power, Inc.",19545,IPP Non-CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,11390.0,NaN
71472,NaN,2021,WY,Wygen II,56319,Black Hills Power Inc,19545,"Black Hills Power, Inc.",19545,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,11391.0,NaN
71473,NaN,2021,WY,Wygen III,56596,Black Hills Power Inc,19545,"Black Hills Power, Inc.",19545,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,11392.0,NaN


In [4]:
master_df['YEAR'].unique()

array([2020, 2018, 2019, 2023, 2022, 2021])

In [5]:
master_df = master_df[master_df['BACODE'] == 'PJM']

**Variables that could be used**
- NAMEPCAP --> Plant nameplate capacity (MW) measures total capacity
- CAPFAC --> capacity factor measures how often the plant will run year round.
- PLPRMFL --> Plant primary fuel
- PLNGENAN --> Plant annual net generation (MWh)
- PLFUELCT --> Plant primary fuel category
- SECTOR --> Plant-level sector.
- NBFACTOR --> Nonbaseload Factor if ran less. It may demonstrate an increase in efficiency (since they only generate energy unmet by baseload)
- ORISPL --> DOE/EIA ORIS plant or facility code for unique classifer
- PLHTIAN --> Plant annual heat input from combustion (MMBtu). This will tell how much fuel was used/burned during generation. 
- PLCO2AN --> Plant annual CO2 emissions (tons)
- PLCO2RTA --> Plant annual CO2 total output emission rate (lb/MWh)


## Data Cleaning

In [6]:
# filter to relevant states, fuel type, & capacity
pjm_states = ['IL','IN','KY','MI','NC','OH','TN','VA','WV']
fossil_generators = ['OIL', 'GAS', 'COAL']
master_df = master_df[master_df['PSTATABB'].isin(pjm_states) & master_df['PLFUELCT'].isin(fossil_generators)].reset_index(drop=True)
# drops plants that don't generate energy from the grid (negative generation)
master_df = master_df[~master_df['PLNGENAN'].astype(str).str.contains(r'\(', na=False)]
master_df['PSTATABB'].unique()

array(['IL', 'IN', 'KY', 'MI', 'NC', 'OH', 'TN', 'VA', 'WV'], dtype=object)

In [7]:
master_df

,SEQPLT20,YEAR,PSTATABB,PNAME,ORISPL,OPRNAME,OPRCODE,UTLSRVNM,UTLSRVID,SECTOR,...,BIOCO2E,CHPCO2E,PLGENATO,PLGENACO,PLTOPR,PLCOPR,SEQPLT22,CAMDFLAG,SEQPLT,Unnamed: 141
0,3785.0,2020,IL,1515 S Caron Road,7770,Rochelle Municipal Utilities,16179,Rochelle Municipal Utilities,16179,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3787.0,2020,IL,Adkins Energy LLC,58136,Commonwealth Edison Co,4110,Adkins Energy LLC,58089,Industrial CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3798.0,2020,IL,Argonne National Laboratory CHP,60742,Commonwealth Edison Co,4110,Argonne National Laboratory,60444,IPP CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3799.0,2020,IL,Aurora,55279,Commonwealth Edison Co,4110,Aurora Generation LLC,60503,IPP Non-CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3815.0,2020,IL,BP Naperville Cogeneration Facility,50722,City of Naperville - (IL),13208,BP America Inc,2836,Commercial CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1704,NaN,2021,WV,Morgantown Energy Facility,10743,Monongahela Power Co,12796,Morgantown Energy Associates,12949,IPP CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,11314.0,NaN
1705,NaN,2021,WV,Mount Storm Power Station,3954,Virginia Electric & Power Co,19876,Virginia Electric & Power Co,19876,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,11315.0,NaN
1706,NaN,2021,WV,Mountaineer (1301),6264,Appalachian Power Co,733,Appalachian Power Co,733,Electric Utility,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,11316.0,NaN
1707,NaN,2021,WV,"Pleasants Energy, LLC",55349,Monongahela Power Co,12796,Pleasants Energy LLC,55749,IPP Non-CHP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,11323.0,NaN


In [8]:
variables_interest = ['ORISPL','YEAR','PSTATABB','NAMEPCAP','CAPFAC',
                      'SECTOR','PLFUELCT', 'PLNGENAN', 'PLHTIAN', 'PLCO2AN', 'PLCO2RTA']
filtered_df = master_df[variables_interest].copy()
neighbors = ['KY','NC','WV']

def string_to_int(series):
    return (series
        .str.replace(',','')
        .astype(float)
        .fillna(0)
           )

for i in ['NAMEPCAP','PLNGENAN', 'PLHTIAN', 'PLCO2AN', 'PLCO2RTA']:
    filtered_df[i] = string_to_int(filtered_df[i])
    
# filtered_df['PLCO2AN'] = (
#     filtered_df['PLCO2AN']
#     .str.replace(',','')
#     .astype(float)
#     .fillna(0)
#     .astype('int64'))
# filtered_df['PLCO2RTA'] = (
#     filtered_df['PLCO2RTA']
#         .str.replace(',','')
#         .astype(float)
#         .fillna(0)
#         .astype('int64'))
filtered_df = filtered_df[filtered_df['NAMEPCAP'] >= 25]
filtered_df['TREATED'] = (filtered_df['PSTATABB'] == 'VA').astype(int)
filtered_df['NEIGHBOR'] = (filtered_df['PSTATABB'].isin(neighbors)).astype(int)
filtered_df['POST'] = (filtered_df['YEAR'] >= 2021).astype(int)
filtered_df

,ORISPL,YEAR,PSTATABB,NAMEPCAP,CAPFAC,SECTOR,PLFUELCT,PLNGENAN,PLHTIAN,PLCO2AN,PLCO2RTA,TREATED,NEIGHBOR,POST
3,55279,2020,IL,1086.2,0.0808,IPP Non-CHP,GAS,768846.0,7770819.0,461804.0,1201.292,0,0,0
6,55296,2020,IL,312.8,0.0053,IPP Non-CHP,GAS,14431.0,155609.0,9039.0,1252.678,0,0,0
9,55188,2020,IL,611.2,0.3719,IPP Non-CHP,GAS,1990983.0,13538133.0,804563.0,808.206,0,0,0
10,63931,2020,IL,1300.0,NaN,IPP Non-CHP,GAS,0.0,0.0,0.0,0.000,0,0,0
11,55253,2020,IL,356.0,0.0122,IPP Non-CHP,GAS,37942.0,500498.0,29739.0,1567.619,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1704,10743,2021,WV,68.9,NaN,IPP CHP,GAS,0.0,0.0,0.0,0.000,0,1,1
1705,3954,2021,WV,1680.9,0.3741,Electric Utility,COAL,5508707.0,57367773.0,5885931.0,2136.956,0,1,1
1706,6264,2021,WV,1300.0,0.6263,Electric Utility,COAL,7132758.0,71684682.0,7353642.0,2061.935,0,1,1
1707,55349,2021,WV,344.0,0.4156,IPP Non-CHP,GAS,1252463.0,13022037.0,774961.0,1237.499,0,1,1


In [9]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 955 entries, 3 to 1708
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ORISPL    955 non-null    int64  
 1   YEAR      955 non-null    int64  
 2   PSTATABB  955 non-null    object 
 3   NAMEPCAP  955 non-null    float64
 4   CAPFAC    852 non-null    float64
 5   SECTOR    955 non-null    object 
 6   PLFUELCT  955 non-null    object 
 7   PLNGENAN  955 non-null    float64
 8   PLHTIAN   955 non-null    float64
 9   PLCO2AN   955 non-null    float64
 10  PLCO2RTA  955 non-null    float64
 11  TREATED   955 non-null    int64  
 12  NEIGHBOR  955 non-null    int64  
 13  POST      955 non-null    int64  
dtypes: float64(6), int64(5), object(3)
memory usage: 111.9+ KB


In [10]:
# checking how many null without subtracting
print(filtered_df['CAPFAC'].isnull().sum())

103


separate view just to see --------


In [11]:
capfac = filtered_df[filtered_df['CAPFAC'].isnull()]
capfac['YEAR'].value_counts()

YEAR
2020    35
2019    32
2018    29
2023     3
2022     2
2021     2
Name: count, dtype: int64

------


In [12]:
# filtered_df[filtered_df['PLNGENAN'].isnull()]
filtered_df

,ORISPL,YEAR,PSTATABB,NAMEPCAP,CAPFAC,SECTOR,PLFUELCT,PLNGENAN,PLHTIAN,PLCO2AN,PLCO2RTA,TREATED,NEIGHBOR,POST
3,55279,2020,IL,1086.2,0.0808,IPP Non-CHP,GAS,768846.0,7770819.0,461804.0,1201.292,0,0,0
6,55296,2020,IL,312.8,0.0053,IPP Non-CHP,GAS,14431.0,155609.0,9039.0,1252.678,0,0,0
9,55188,2020,IL,611.2,0.3719,IPP Non-CHP,GAS,1990983.0,13538133.0,804563.0,808.206,0,0,0
10,63931,2020,IL,1300.0,NaN,IPP Non-CHP,GAS,0.0,0.0,0.0,0.000,0,0,0
11,55253,2020,IL,356.0,0.0122,IPP Non-CHP,GAS,37942.0,500498.0,29739.0,1567.619,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1704,10743,2021,WV,68.9,NaN,IPP CHP,GAS,0.0,0.0,0.0,0.000,0,1,1
1705,3954,2021,WV,1680.9,0.3741,Electric Utility,COAL,5508707.0,57367773.0,5885931.0,2136.956,0,1,1
1706,6264,2021,WV,1300.0,0.6263,Electric Utility,COAL,7132758.0,71684682.0,7353642.0,2061.935,0,1,1
1707,55349,2021,WV,344.0,0.4156,IPP Non-CHP,GAS,1252463.0,13022037.0,774961.0,1237.499,0,1,1


In this dataset, an NAN annual plant generation affects Capacity Factor('CAPFAC') because CAPFAC is the ratio of generation throughout the year. Since there is no data for the generation, there is no capacity factor. I will now drop these rows.

In [13]:
filtered_df = filtered_df[~filtered_df['CAPFAC'].isnull()]
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 852 entries, 3 to 1708
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ORISPL    852 non-null    int64  
 1   YEAR      852 non-null    int64  
 2   PSTATABB  852 non-null    object 
 3   NAMEPCAP  852 non-null    float64
 4   CAPFAC    852 non-null    float64
 5   SECTOR    852 non-null    object 
 6   PLFUELCT  852 non-null    object 
 7   PLNGENAN  852 non-null    float64
 8   PLHTIAN   852 non-null    float64
 9   PLCO2AN   852 non-null    float64
 10  PLCO2RTA  852 non-null    float64
 11  TREATED   852 non-null    int64  
 12  NEIGHBOR  852 non-null    int64  
 13  POST      852 non-null    int64  
dtypes: float64(6), int64(5), object(3)
memory usage: 99.8+ KB


In [14]:
filtered_df

,ORISPL,YEAR,PSTATABB,NAMEPCAP,CAPFAC,SECTOR,PLFUELCT,PLNGENAN,PLHTIAN,PLCO2AN,PLCO2RTA,TREATED,NEIGHBOR,POST
3,55279,2020,IL,1086.2,0.0808,IPP Non-CHP,GAS,768846.0,7770819.0,461804.0,1201.292,0,0,0
6,55296,2020,IL,312.8,0.0053,IPP Non-CHP,GAS,14431.0,155609.0,9039.0,1252.678,0,0,0
9,55188,2020,IL,611.2,0.3719,IPP Non-CHP,GAS,1990983.0,13538133.0,804563.0,808.206,0,0,0
11,55253,2020,IL,356.0,0.0122,IPP Non-CHP,GAS,37942.0,500498.0,29739.0,1567.619,0,0,0
13,55438,2020,IL,540.0,0.0800,IPP Non-CHP,GAS,378234.0,4420334.0,262695.0,1389.061,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1703,3948,2021,WV,1632.6,0.3335,Electric Utility,COAL,4769104.0,49696486.0,5098240.0,2138.029,0,1,1
1705,3954,2021,WV,1680.9,0.3741,Electric Utility,COAL,5508707.0,57367773.0,5885931.0,2136.956,0,1,1
1706,6264,2021,WV,1300.0,0.6263,Electric Utility,COAL,7132758.0,71684682.0,7353642.0,2061.935,0,1,1
1707,55349,2021,WV,344.0,0.4156,IPP Non-CHP,GAS,1252463.0,13022037.0,774961.0,1237.499,0,1,1


In [15]:
filtered_df.to_csv('../data/final_dataset.csv', index=False)